In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel,Field

C:\Users\Armaan Dogra\AppData\Local\Temp\ipykernel_33032\1069200175.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
%pip install -q sentence-transformers
## Document Load
loader = PyPDFLoader("../data/medical_report.pdf")
docs = loader.load()

## split - chunks
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000,chunk_overlap = 200)
docs = splitter.split_documents(docs)

## embedings and vector DB
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = InMemoryVectorStore.from_documents(
    documents=docs,
    embedding=embeddings
)


[notice] A new release of pip available: 22.3.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


c:\Users\Armaan Dogra\OneDrive\Desktop\GenAI_Series\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3176.03it/s]


In [4]:
llm = ChatGroq(model='openai/gpt-oss-20b')

In [5]:
class RagState(BaseModel):
    question: str = Field(description="User Question")
    documents:list=[]
    context: str = Field(description="AI context for user question",default="")
    answer : str = Field(description="Final Answer",default="")


In [6]:
# question -> retrieve -> context -> generate -> end

def retrieve_node(state:RagState) -> RagState:
    docs = vector_store.similarity_search(query = state.question)
    state.documents = docs
    return state

def create_context_node(state:RagState)-> RagState:
    context  = ""
    for doc in state.documents:
        context += doc.page_content + "\n\n"
    state.context = context
    return state

def generate_node(state:RagState) -> RagState:
    prompt = f"""
        You are a assistant and provide the answer for user question based on the provided context
        If you dont find the relevant answer, then just say "i dont know".
        Context is {state.context},
        Question is {state.question}
    """
    res = llm.invoke(prompt)
    state.answer = res.content
    return state


In [7]:
graph = StateGraph(RagState)

graph.add_node("retrieve_node",retrieve_node)
graph.add_node("create_context_node",create_context_node)
graph.add_node("generate_node",generate_node)

graph.add_edge(START,"retrieve_node")
graph.add_edge("retrieve_node","create_context_node")
graph.add_edge("create_context_node","generate_node")
graph.add_edge("generate_node",END)

graph = graph.compile()

In [8]:
res = graph.invoke({"question":"What is the name of patient?"})

In [9]:
res["answer"]

'The patient’s name is **Ms. Nikita Chudhary**.'